In [1]:
# Created on Sept 18, 2025
# Created by: Afeena
# Purpose: This file reads the raw file and creates the cleaned file

import pandas as pd
import numpy as np
import os as os

# File Paths
raw_file_path = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\01Wrangle\\TorontoCity_traffic_collisions_KPI_RAW.csv"
modified_file_path = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\01Wrangle\\TorontoCity_traffic_collisions_KPI_Modified.csv"

# Open raw data
df_raw = pd.read_csv(raw_file_path)

#Rename columns to something meaningful
df_raw.rename(columns={
    "ACCNUM": "AccidentNumber",
    "DATE": "DateCollissionOccured",
    "TIME": "TimeCollisionOccured",
    "STREET1": "Street1CollisionOccrred",
    "STREET2": "Street2CollissionOccrred",
    "OFFSET": "DistanceAndDirectionOfCollision",
    "ROAD_CLASS": "RoadClassification",
    "DISTRICT": "CityDistrict",
    "ACCLOC": "CollisionLocation",
    "TRAFFCTL": "TrafficControlType",
    "VISIBILITY": "EnvironmentCondition",
    "LIGHT": "LightingCondition",
    "RDSFCOND": "RoadSurfaceCondition",
    "ACCLASS": "AccidentClassification",
    "IMPACTYPE": "InitalImpactType",
    "INVTYPE": "InvolvementType",
    "INVAGE": "AgeOfInvolvedParty",
    "INJURY": "SeverityOfInjury",
    "FATAL_NO": "FatalSequentialNumber",
    "INITDIR": "InitialDirectionTravel",
    "VEHTYPE": "VehicleType",
    "MANOEUVER": "VehicleManouever",
    "DRIVACT": "ApparentDriverAction",
    "DRIVCOND": "DriverCondition",
    "PEDTYPE": "PedestrianCrashType",
    "PEDACT": "PedestrianAction",
    "PEDCOND": "PederstrianCondition",
    "CYCLISTYPE": "CyclistCrashType",
    "CYCACT": "CyclistAction",
    "CYCCOND": "CyclistCondition",
    "PEDESTRIAN": "PedestrianInvolvedInCollision",
    "CYCLIST": "CyclistInvolvedInCollision",
    "AUTOMOBILE": "DriverInvolvedInCollision",
    "MOTORCYCLE": "MotorcyclistInvolvedInCollision",
    "TRUCK": "TruckDriverInvolvedInCrash",
    "TRSN_CITY_VEH": "TransitOrCityVehicleInvolvedInCollision",
    "EMERG_VEH": "EmergencyVehicleInvolvedInCollision",
    "PASSENGER": "PassengerInvolvedInCollision",
    "SPEEDING": "SpeedRelatedCollision",
    "AG_DRIV": "AggressiveDistractedDriivingCollision",
    "REDLIGHT": "RedLightRelatedCollision",
    "ALCOHOL": "AlcoholLightRelatedCollision",
    "DISABILITY": "DisabilityRelatedCollision",
    "HOOD_158": "ID_TorontorNeighbourhood_NEW",
    "NEIGHBOURHOOD_158": "TorontoNeighbourhood_NEW",
    "HOOD_140": "ID_TorontoNeighbourhood_OLD",
    "NEIGHBOURHOOD_140": "TorontoNeighbourhood_OLD",
    "DIVISION": "TorontoPoliceDivision"
}, inplace=True)

# Identify columns of interest
cols_of_interest= ["SeverityOfInjury", "InvolvementType","AgeOfInvolvedParty",
                #    "CityDistrict","TorontoNeighbourhood_NEW",
                   "TimeCollisionOccured","DateCollissionOccured",
                   "LightingCondition","RoadSurfaceCondition"]

# assign the columns of interest that have missing values to a variable
missing_in_cols_of_interest = df_raw[cols_of_interest].isnull().sum()[df_raw[cols_of_interest].isnull().sum() > 0]
print("Columns of interest with missing values and their counts:\n", missing_in_cols_of_interest)

#check uniques values of each column in missing_in_cols_of_interest
for col in missing_in_cols_of_interest.index:
    unique_values = df_raw[col].unique()
    print(f"\n Unique values in '{col}': {unique_values}")

# drop records
df_clean = df_raw.dropna(subset=["SeverityOfInjury", "AgeOfInvolvedParty",
                                # "CityDistrict", "TorontoNeighbourhood_NEW",
                                "TimeCollisionOccured", "DateCollissionOccured",
                                "RoadSurfaceCondition"]).copy()
print("\n Shape of the original dataset:", df_raw.shape)
print("\n Shape of the cleaned dataset:", df_clean.shape)

# Replace the nulls for the following columns that have "Other": InvolvementType, LightingCondition
df_clean.fillna({"InvolvementType": "Missing"})
df_clean.fillna({"LightingCondition": "Missing"})

# Check missing values in DateCollissionOccured and TimeCollisionOccured
# print("\n Missing values in 'DateCollissionOccured':", df_raw["DateCollissionOccured"].isnull().sum())
# print("Missing values in 'TimeCollisionOccured':", df_raw["TimeCollisionOccured"].isnull().sum())

#Create new columns
df_clean["MONTH_NUMBER_COLLISION_OCCURED"] = pd.to_datetime(df_clean["DateCollissionOccured"]).dt.month
df_clean["MONTH_NAME_COLLISION_OCCURED"] = pd.to_datetime(df_clean["DateCollissionOccured"]).dt.month_name()
df_clean["YEAR_COLLISION_OCCURED"] = pd.to_datetime(df_clean["DateCollissionOccured"]).dt.year

# Convert TIME to 24HRS format

# check unique lengths of TimeCollisionOccured
# print(df_raw["TimeCollisionOccured"].astype(str).apply(len).unique())

# pad string to make it 4 characters
df_clean["TIME_COLLISION_OCCURED24HRS"] = df_clean["TimeCollisionOccured"].astype(str).str.zfill(4)

# convert to 24HRformat
df_clean["TIME_COLLISION_OCCURED24HRS"] = pd.to_datetime(df_clean["TIME_COLLISION_OCCURED24HRS"], format='%H%M').dt.strftime('%H:%M')
# print(df_raw[["DateCollissionOccured","TimeCollisionOccured","MONTH_COLLISION_OCCURED","YEAR_COLLISION_OCCURED","TIME_COLLISION_OCCURED24HRS"]].head())

# Create PEAKOFFPEAK column using TIME_COLLISION_OCCURED24HRS
df_clean["PEAKOFFPEAK"] = np.where(
    ((df_clean["TIME_COLLISION_OCCURED24HRS"] >= "06:00") & (df_clean["TIME_COLLISION_OCCURED24HRS"] <= "09:00")) |
    ((df_clean["TIME_COLLISION_OCCURED24HRS"] >= "15:00") & (df_clean["TIME_COLLISION_OCCURED24HRS"] <= "19:00")) |
    ((df_clean["TIME_COLLISION_OCCURED24HRS"] >= "18:00") & (df_clean["TIME_COLLISION_OCCURED24HRS"] <= "21:00")),
    "PEAK", "OFF-PEAK"
)

df_clean["SEASON"] = np.where(
    ((df_clean["MONTH_NAME_COLLISION_OCCURED"] == "December") | (df_clean["MONTH_NAME_COLLISION_OCCURED"] == "January") | (df_clean["MONTH_NAME_COLLISION_OCCURED"] == "February")),
   "WINTER", "NOT-WINTER"
)

# Create hour collision occured using TimeCollisionOccured
df_clean["HOUR_COLLISION_OCCURED"] = pd.to_datetime(df_clean["TimeCollisionOccured"]).dt.hour
# print(df_raw[["TimeCollisionOccured","HOUR_COLLISION_OCCURED"]].head())

# Convert TIME_COLLISION_OCCURED24HRS to hour as integer
df_clean["HOUR_COLLISION_OCCURED"] = pd.to_datetime(df_clean["TIME_COLLISION_OCCURED24HRS"], format="%H:%M").dt.hour

#Function to create TIME_RANGE column
def add_time_range_column(whichDataframe):
    def determine_time_range(time_str):
        time = pd.to_datetime(time_str, format='%H:%M').time()
        if pd.to_datetime('00:00', format='%H:%M').time() <= time <= pd.to_datetime('02:59', format='%H:%M').time():
            return 'Midnight to 2:59'
        elif pd.to_datetime('03:00', format='%H:%M').time() <= time <= pd.to_datetime('05:59', format='%H:%M').time():
            return '3:00 to 5:59'
        elif pd.to_datetime('06:00', format='%H:%M').time() <= time <= pd.to_datetime('08:59', format='%H:%M').time():
            return '6:00 to 8:59'
        elif pd.to_datetime('09:00', format='%H:%M').time() <= time <= pd.to_datetime('11:59', format='%H:%M').time():
            return '9:00 to 11:59'
        elif pd.to_datetime('12:00', format='%H:%M').time() <= time <= pd.to_datetime('14:59', format='%H:%M').time():
            return '12:00 to 14:59'
        elif pd.to_datetime('15:00', format='%H:%M').time() <= time <= pd.to_datetime('17:59', format='%H:%M').time():
            return '15:00 to 17:59'
        elif pd.to_datetime('18:00', format='%H:%M').time() <= time <= pd.to_datetime('20:59', format='%H:%M').time():
            return '18:00 to 20:59'
        elif pd.to_datetime('21:00', format='%H:%M').time() <= time <= pd.to_datetime('23:59', format='%H:%M').time():
            return '21:00 to 23:59'
        return 'Unknown'

    whichDataframe['TIME_RANGE'] = whichDataframe['TIME_COLLISION_OCCURED24HRS'].apply(determine_time_range)
    return whichDataframe

#Call function to create TIME_RANGE column
add_time_range_column(df_clean)

#Make Dummies for Categorical columns
df_UsrInvTypeDummies = pd.get_dummies(df_clean.InvolvementType, prefix="UserInvType").drop(columns="UserInvType_Driver").astype(int)
df_UsrInvTypeDummies.columns = df_UsrInvTypeDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_UsrAgeRangeDummies = pd.get_dummies(df_clean.AgeOfInvolvedParty, prefix="UserAgeRange").drop(columns="UserAgeRange_20 to 24").astype(int)
df_UsrAgeRangeDummies.columns = df_UsrAgeRangeDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_EnvLightingConditionDummies = pd.get_dummies(df_clean.LightingCondition, prefix="EnvLightingCondition").drop(columns="EnvLightingCondition_Daylight").astype(int)
df_EnvLightingConditionDummies.columns = df_EnvLightingConditionDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_EnvRoadSurfaceConditionDummies = pd.get_dummies(df_clean.RoadSurfaceCondition, prefix="EnvRoadSurfaceCondition").drop(columns="EnvRoadSurfaceCondition_Dry").astype(int)
df_EnvRoadSurfaceConditionDummies.columns = df_EnvRoadSurfaceConditionDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_TimeYearDummies = pd.get_dummies(df_clean.YEAR_COLLISION_OCCURED, prefix="TimeYear").drop(columns="TimeYear_2006").astype(int)
df_TimeYearDummies.columns = df_TimeYearDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_TimeMonthDummies = pd.get_dummies(df_clean.MONTH_NAME_COLLISION_OCCURED, prefix="TimeMonth").drop(columns="TimeMonth_January").astype(int)
df_TimeMonthDummies.columns = df_TimeMonthDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_TimeHourDummies = pd.get_dummies(df_clean.HOUR_COLLISION_OCCURED, prefix="TimeHour").drop(columns="TimeHour_12").astype(int)
df_TimeHourDummies.columns = df_TimeHourDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_TimePeakOffPeakDummies = pd.get_dummies(df_clean.PEAKOFFPEAK, prefix="TimePeakOffPeak").drop(columns="TimePeakOffPeak_PEAK").astype(int)
df_TimePeakOffPeakDummies.columns = df_TimePeakOffPeakDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_TimeRangeDummies = pd.get_dummies(df_clean.TIME_RANGE, prefix="TimeRange").drop(columns="TimeRange_12:00 to 14:59").astype(int)
df_TimeRangeDummies.columns = df_TimeRangeDummies.columns.str.replace(r'[ ,:\-\/]', '_', regex=True)

df_TimeSeasonDummies = pd.get_dummies(df_clean.SEASON, prefix="TimeSeason").drop(columns="TimeSeason_NOT-WINTER").astype(int)
df_TimeSeasonDummies.columns = df_TimeSeasonDummies.columns.str.replace(r'[ ,\-\/]', '_', regex=True)

df_SeverityOfInjuryDummies = pd.get_dummies(df_clean.SeverityOfInjury, prefix="SeverityOfInjury").drop(columns="SeverityOfInjury_Minimal").astype(int)


myDataCombined = pd.concat([df_clean, 
                            df_UsrInvTypeDummies,
                            df_UsrAgeRangeDummies,
                            df_EnvLightingConditionDummies,
                            df_EnvRoadSurfaceConditionDummies,
                            df_TimeYearDummies,
                            df_TimeMonthDummies,
                            df_TimeHourDummies,
                            df_TimePeakOffPeakDummies,
                            df_TimeRangeDummies,
                            df_TimeSeasonDummies,
                            df_SeverityOfInjuryDummies
                            ], axis = 1)

print("Categorical dummies were added!!!!")

#Write the modified file exclude the index
myDataCombined.to_csv(modified_file_path, index=False)

# # Open raw data & show new size
df_modifiedFinal  = pd.read_csv(modified_file_path, low_memory=False)
print("New size:", df_modifiedFinal.shape)


Columns of interest with missing values and their counts:
 SeverityOfInjury        8897
InvolvementType           16
LightingCondition          4
RoadSurfaceCondition      29
dtype: int64

 Unique values in 'SeverityOfInjury': ['Major' 'Minor' nan 'Fatal' 'Minimal']

 Unique values in 'InvolvementType': ['Passenger' 'Driver' 'Vehicle Owner' 'Other Property Owner' 'Pedestrian'
 'Cyclist' 'Other' 'Motorcycle Driver' 'Truck Driver' 'In-Line Skater'
 'Driver - Not Hit' 'Motorcycle Passenger' nan 'Moped Driver' 'Wheelchair'
 'Pedestrian - Not Hit' 'Trailer Owner' 'Witness' 'Cyclist Passenger'
 'Moped Passenger']

 Unique values in 'LightingCondition': ['Dark' 'Dark, artificial' 'Daylight' 'Dusk' 'Dawn' 'Dusk, artificial'
 'Dawn, artificial' 'Daylight, artificial' 'Other' nan]

 Unique values in 'RoadSurfaceCondition': ['Wet' 'Slush' 'Dry' 'Ice' 'Loose Snow' 'Other' 'Packed Snow'
 'Spilled liquid' 'Loose Sand or Gravel' nan]

 Shape of the original dataset: (18957, 49)

 Shape of the cleaned